In [ ]:
import numpy as np
from PIL import Image, ImageCms
from math import sqrt, atan2, sin, degrees, radians, pi, cos, exp
import pandas as pd
import time

# Test Different ways to Calcuate DeltaE2000 Values

### Setup (Always Run)

In [2]:
from NathanClustering import Pixel

im = Image.open("Images/Madonna.JPG")

#Covert RGB to CIELAB 
srgbProfile = ImageCms.createProfile("sRGB")
labProfile  = ImageCms.createProfile("LAB")
rgb2lab = ImageCms.buildTransformFromOpenProfiles(srgbProfile, labProfile, "RGB", "LAB")
lab = ImageCms.applyTransform(im, rgb2lab)


allPixels = im.load()
openedImage = np.asarray(im)
openedImageLAB = np.asarray(lab)

### Calculate distance at a time

### Calculate all at once using Pandas

In [5]:
import pandas as pd

coordColor = pd.DataFrame(openedImageLAB.reshape(openedImageLAB.shape[0] * openedImageLAB.shape[1], 3), columns=["L", "A", "B"])
coordColor["X"] = coordColor.index % openedImage.shape[1]
coordColor["Y"] = coordColor.index // openedImage.shape[1]
coordColor = coordColor.astype({"X":pd.UInt16Dtype(), "Y":pd.UInt16Dtype()})

# Create DataFrame [(X1,Y1),(X2,Y2), Color1, Color2] where X & Y are touching
    #(X1,Y1),(X2,Y2) where euclidDist = 1
# Calcuate metric row-wise for each combo
#coordColor[:10].merge(coordColor[:10], how="cross")

"""
#Cross Join is too much data at once. So will concat cross-joined subsets (Need more than 2 halves)
    #Find middle row, create subsets that has row in both subsets. This will negate the dividing line between subsets
firstHalf = coordColor[:coordColor.shape[0]//2]
secondHalf = coordColor[coordColor.shape[0]//2 - openedImageLAB.shape[1]:]
    #Now we cross join each half with itself and subset correctly
firstHalf = firstHalf.merge(firstHalf, how="cross")
firstHalf = firstHalf[((firstHalf["X_x"]-firstHalf["X_y"]) + (firstHalf["Y_x"]-firstHalf["Y_y"])) == 1]
firstHalf


# Merge when X=X or when Y=Y. Since looking at adjacent pixels, one of x's or y's will be equal
mergeX = pd.merge(coordColor.rename(columns={"X":"X_x"})[1:10],coordColor.rename(columns={"X":"X_y"})[1:10], left_on="X_x", right_on="X_y")
mergeX

mergeY = pd.merge(coordColor.rename(columns={"Y":"Y_x"})[:4032*6],
                  coordColor.rename(columns={"Y":"Y_y"})[:4032*6], left_on="Y_x", right_on="Y_y")
mergeY
"""


,L_x,A_x,B_x,X_x,Y_x,L_y,A_y,B_y,X_y,Y_y
0,169,254,221,0,0,169,254,221,0,0
1,169,254,221,0,0,168,254,221,1,0
2,169,254,221,0,0,168,254,221,2,0
3,169,254,221,0,0,168,254,221,3,0
4,169,254,221,0,0,169,254,221,4,0
...,...,...,...,...,...,...,...,...,...,...
97542139,165,254,224,4031,5,166,253,225,4027,5
97542140,165,254,224,4031,5,165,254,224,4028,5
97542141,165,254,224,4031,5,165,254,224,4029,5
97542142,165,254,224,4031,5,165,254,224,4030,5


In [16]:
df = pd.DataFrame(columns=["X1", "Y1", "L1", "A1", "B1", "X2", "Y2", "L2", "A2", "B2"])

#Iterate through all pixels, and append each of its four adjacent to df using below
listy = []
for row in range(openedImageLAB.shape[0]):
    for col in range(openedImageLAB.shape[1]):
        if col - 1 >= 0: #Left
            listy.append([col, row, openedImageLAB[row][col][0], openedImageLAB[row][col][1], openedImageLAB[row][col][2],
                          col-1, row, openedImageLAB[row][col-1][0], openedImageLAB[row][col-1][1], openedImageLAB[row][col-1][2]])
        if col + 1 < openedImageLAB.shape[1]: #Right
            listy.append([col, row, openedImageLAB[row][col][0], openedImageLAB[row][col][1], openedImageLAB[row][col][2],
                          col+1, row, openedImageLAB[row][col+1][0], openedImageLAB[row][col+1][1], openedImageLAB[row][col+1][2]])
        if row - 1 >= 0: #Up
            listy.append([col, row, openedImageLAB[row][col][0], openedImageLAB[row][col][1], openedImageLAB[row][col][2],
                          col, row-1, openedImageLAB[row-1][col][0], openedImageLAB[row-1][col][1], openedImageLAB[row-1][col][2]])
        if row + 1 < openedImageLAB.shape[0]: #Down
            listy.append([col, row, openedImageLAB[row][col][0], openedImageLAB[row][col][1], openedImageLAB[row][col][2],
                          col, row+1, openedImageLAB[row+1][col][0], openedImageLAB[row+1][col][1], openedImageLAB[row+1][col][2]])

#Or all four can be appended/concatenated at once
pd.DataFrame(listy, columns=["X1", "Y1", "L1", "A1", "B1", "X2", "Y2", "L2", "A2", "B2"])

KeyboardInterrupt: 

In [18]:
listy[40780326]

[426, 2529, 131, 254, 17, 427, 2529, 121, 254, 17]

In [ ]:
class QuickSort:

    def __init__(self, array: list):
        self.low = 0
        self.high = len(array) - 1
        self.sort(array)

    def partition(self, arr): 
        i = (self.low - 1 ) 
        x = arr[self.high][0] 
        for j in range(self.low, self.high): 
            if   arr[j][0] >= x: 
                i = i + 1
                arr[i], arr[j] = arr[j], arr[i] 
        arr[i + 1], arr[self.high] = arr[self.high], arr[i + 1] 
        return (i + 1) 

    def sort(self, array): 
        size = self.high - self.low + 1
        stack = [0] * (size) 
        top = -1
        top = top + 1
        stack[top] = self.low 
        top = top + 1
        stack[top] = self.high 
        while top >= 0: 
            self.high = stack[top] 
            top = top - 1
            self.low = stack[top] 
            top = top - 1
            p = self.partition(array) 
            if p - 1 > self.low: 
                top = top + 1
                stack[top] = self.low 
                top = top + 1
                stack[top] = p - 1
            if p + 1 < self.high: 
                top = top + 1
                stack[top] = p + 1
                top = top + 1
                stack[top] = self.high 
